In [3]:
# Cell 1: Setup
!pip install -q sentence-transformers vstash
!git clone https://github.com/stffns/vstash.git /content/vstash 2>/dev/null || (cd /content/vstash && git pull origin develop)
%cd /content/vstash

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.4/163.4 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 25.6 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.8/324.8 kB 33.1 MB/s eta 0:00:00
From https://github.com/stffns/vstash
 * branch            develop    -> FETCH_HEAD
Already up to date.
/content/vstash


In [7]:
# Cell 2: Generate triples for BGE-base (768d) - different disagreements than BGE-small
# Each model has its own blind spots, so triples must be regenerated per model
import random
import tempfile
from pathlib import Path

import sys
import os

os.chdir("/content/vstash")
sys.path.insert(0, ".")
from experiments.beir_benchmark import download_beir, load_beir
from vstash.store import VstashStore

DATASETS = ["scifact", "nfcorpus", "fiqa"]
TOP_K = 10


def generate_triples_for_model(model_name, dim):
    from sentence_transformers import SentenceTransformer

    st_model = SentenceTransformer(model_name)

    all_pairs = []
    for ds_name in DATASETS:
        cache = download_beir(ds_name)
        corpus, queries, qrels = load_beir(cache)
        doc_items = list(corpus.items())[:5000]

        db_path = tempfile.mktemp(suffix=".db")
        store = VstashStore(db_path, embedding_dim=dim)

        # Ingest with this model's embeddings
        doc_id_to_path = {}
        doc_texts = {}
        texts_batch = []
        ids_batch = []
        for doc_id, doc in doc_items:
            text = (doc.get("title", "") + "\n" + doc.get("text", "")).strip()[:512]
            texts_batch.append(text)
            ids_batch.append(doc_id)

        all_embs = st_model.encode(texts_batch, show_progress_bar=True, batch_size=256)

        batch = []
        for doc_id, text, emb in zip(ids_batch, texts_batch, all_embs):
            path = f"/{ds_name}/{doc_id}"
            doc_id_to_path[doc_id] = path
            doc_texts[path] = text
            batch.append(
                {
                    "path": path,
                    "title": corpus[doc_id].get("title", doc_id),
                    "chunks": [text],
                    "embeddings": [emb.tolist()],
                    "source_type": "text",
                }
            )
            if len(batch) >= 500:
                store.add_documents_batch(batch)
                batch = []
        if batch:
            store.add_documents_batch(batch)

        query_items = [(qid, qt) for qid, qt in queries.items() if qid in qrels]
        pairs = []
        disagree = 0

        for qid, qtext in query_items:
            gold_paths = {
                doc_id_to_path[d] for d in qrels[qid] if qrels[qid][d] > 0 and d in doc_id_to_path
            }
            if not gold_paths:
                continue

            qemb = st_model.encode(qtext).tolist()

            try:
                vec_r = store.search(
                    query_embedding=qemb,
                    query_text=qtext,
                    top_k=TOP_K,
                    vec_weight=0.95,
                    fts_weight=0.05,
                    adaptive_rrf=False,
                )
                fts_r = store.search(
                    query_embedding=qemb,
                    query_text=qtext,
                    top_k=TOP_K,
                    vec_weight=0.05,
                    fts_weight=0.95,
                    adaptive_rrf=False,
                )
            except Exception:
                continue

            vec_paths = {r.path for r in vec_r[:5]}
            fts_paths = {r.path for r in fts_r[:5]}
            if vec_paths != fts_paths:
                disagree += 1

            result_texts = {r.path: r.text for r in vec_r + fts_r}

            for gp in gold_paths:
                if gp not in result_texts:
                    continue
                pos = result_texts[gp]
                hard_neg = None
                for r in vec_r[:5]:
                    if r.path not in fts_paths and r.path not in gold_paths:
                        hard_neg = r.text
                        break
                if hard_neg is None:
                    for r in fts_r[:5]:
                        if r.path not in vec_paths and r.path not in gold_paths:
                            hard_neg = r.text
                            break
                pairs.append({"query": qtext, "positive": pos, "negative": hard_neg})

        n_q = len(
            [
                q
                for q, _ in query_items
                if any(qrels[q].get(d, 0) > 0 and d in doc_id_to_path for d in qrels[q])
            ]
        )
        print(
            f"  {ds_name}: {n_q} queries, {disagree} disagree ({disagree / max(n_q, 1) * 100:.1f}%), {len(pairs)} pairs"
        )
        all_pairs.extend(pairs)
        store.close()
        Path(db_path).unlink(missing_ok=True)

    random.Random(42).shuffle(all_pairs)
    return all_pairs


print("Generating triples for BGE-base (768d)...")
base_pairs = generate_triples_for_model("BAAI/bge-base-en-v1.5", 768)
print(f"Total BGE-base triples: {len(base_pairs)}")

Generating triples for BGE-base (768d)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

  scifact: 295 queries, 189 disagree (64.1%), 272 pairs


Batches:   0%|          | 0/15 [00:00<?, ?it/s]

  nfcorpus: 323 queries, 221 disagree (68.4%), 991 pairs


Batches:   0%|          | 0/20 [00:00<?, ?it/s]

  fiqa: 135 queries, 120 disagree (88.9%), 108 pairs
Total BGE-base triples: 1371


In [8]:
# Cell 3: Train BGE-base with MNRL + explicit hard negatives
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader

model = SentenceTransformer("BAAI/bge-base-en-v1.5")

examples = []
for p in base_pairs:
    if p.get("negative"):
        examples.append(InputExample(texts=[p["query"], p["positive"], p["negative"]]))
    else:
        examples.append(InputExample(texts=[p["query"], p["positive"]]))

loader = DataLoader(examples, shuffle=True, batch_size=64)
loss = losses.MultipleNegativesRankingLoss(model)

print(f"Training BGE-base: {len(examples)} examples, 2 epochs")
model.fit(
    train_objectives=[(loader, loss)],
    epochs=2,
    warmup_steps=50,
    optimizer_params={"lr": 3e-6},
    output_path="experiments/models/bge-base-rrf",
    show_progress_bar=True,
)
model.save("experiments/models/bge-base-rrf")
print("BGE-base training done.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Training BGE-base: 1371 examples, 2 epochs


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

BGE-base training done.


In [9]:
# Cell 4: Evaluate all models head-to-head
import math
import statistics
from sentence_transformers import SentenceTransformer

import sys

sys.path.insert(0, "/content/vstash")

BASELINES = {
    "scifact": {"BM25": 0.665, "ColBERTv2": 0.693},
    "nfcorpus": {"BM25": 0.325, "ColBERTv2": 0.344},
    "fiqa": {"BM25": 0.236, "ColBERTv2": 0.356},
    "scidocs": {"BM25": 0.158, "ColBERTv2": 0.154},
    "arguana": {"BM25": 0.315, "ColBERTv2": 0.463},
}

MODELS = {
    "BGE-small base": ("BAAI/bge-small-en-v1.5", 384),
    "BGE-small tuned (v5)": ("Stffens/bge-small-rrf-v2", 384),
    "BGE-base base": ("BAAI/bge-base-en-v1.5", 768),
    "BGE-base tuned": ("experiments/models/bge-base-rrf", 768),
}

EVAL_DATASETS = ["scifact", "nfcorpus", "fiqa", "scidocs", "arguana"]


def ndcg_at_k(ranked_ids, qr, k=10):
    gains = [qr.get(did, 0) for did in ranked_ids[:k]]
    ideal = sorted(qr.values(), reverse=True)[:k]
    idcg = sum(r / math.log2(i + 2) for i, r in enumerate(ideal))
    dcg = sum(g / math.log2(i + 2) for i, g in enumerate(gains[:k]))
    return dcg / idcg if idcg > 0 else 0.0


all_results = {}
for ds_name in EVAL_DATASETS:
    cache = download_beir(ds_name)
    corpus, queries, qrels = load_beir(cache)
    test_qids = list(qrels.keys())
    print(f"\n  {ds_name}: {len(corpus)} docs, {len(test_qids)} queries")
    all_results[ds_name] = {}

    for label, (model_path, dim) in MODELS.items():
        st_model = SentenceTransformer(model_path)
        db_path = tempfile.mktemp(suffix=".db")
        store = VstashStore(db_path, embedding_dim=dim)

        doc_ids = list(corpus.keys())
        doc_texts = [
            (corpus[x].get("title", "") + "\n" + corpus[x].get("text", "")).strip()[:512]
            for x in doc_ids
        ]
        all_embs = st_model.encode(doc_texts, show_progress_bar=False, batch_size=256)

        doc_id_map = {}
        batch = []
        for doc_id, text, emb in zip(doc_ids, doc_texts, all_embs):
            path = f"/beir/{doc_id}"
            doc_id_map[path] = doc_id
            batch.append(
                {
                    "path": path,
                    "title": corpus[doc_id].get("title", doc_id),
                    "chunks": [text],
                    "embeddings": [emb.tolist()],
                    "source_type": "text",
                }
            )
            if len(batch) >= 500:
                store.add_documents_batch(batch)
                batch = []
        if batch:
            store.add_documents_batch(batch)

        ndcgs = []
        for qid in test_qids:
            qemb = st_model.encode(queries[qid]).tolist()
            results = store.search(query_embedding=qemb, query_text=queries[qid], top_k=10)
            ranked = [doc_id_map.get(r.path, "") for r in results]
            ndcgs.append(ndcg_at_k(ranked, qrels[qid], 10))

        mean_ndcg = statistics.mean(ndcgs)
        print(f"    {label:>25}: NDCG@10={mean_ndcg:.4f}")
        all_results[ds_name][label] = mean_ndcg
        store.close()
        Path(db_path).unlink(missing_ok=True)

# Summary
print(f"\n{'=' * 100}")
print("  MULTI-MODEL COMPARISON")
print(f"{'=' * 100}")
print(
    f"{'Dataset':>12} {'ColBERTv2':>10} {'sm base':>10} {'sm tuned':>10} {'base base':>10} {'base tuned':>10}"
)
print(f"  {'-' * 85}")
for ds in EVAL_DATASETS:
    col = BASELINES[ds]["ColBERTv2"]
    sm_b = all_results[ds].get("BGE-small base", 0)
    sm_t = all_results[ds].get("BGE-small tuned (v5)", 0)
    ba_b = all_results[ds].get("BGE-base base", 0)
    ba_t = all_results[ds].get("BGE-base tuned", 0)
    print(f"{ds:>12} {col:>10.3f} {sm_b:>10.4f} {sm_t:>10.4f} {ba_b:>10.4f} {ba_t:>10.4f}")

print("\nKey question: Does BGE-small tuned (33M) beat BGE-base untrained (110M)?")
for ds in EVAL_DATASETS:
    sm_t = all_results[ds].get("BGE-small tuned (v5)", 0)
    ba_b = all_results[ds].get("BGE-base base", 0)
    delta = (sm_t - ba_b) / ba_b * 100 if ba_b > 0 else 0
    winner = "sm tuned WINS" if sm_t > ba_b else "base base wins"
    print(f"  {ds:>12}: sm_tuned={sm_t:.4f} vs base_base={ba_b:.4f} ({delta:+.1f}%) -> {winner}")


  scifact: 5183 docs, 300 queries


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


                BGE-base base: NDCG@10=0.4220


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

               BGE-base tuned: NDCG@10=0.4291

  MULTI-MODEL COMPARISON
     Dataset  ColBERTv2    sm base   sm tuned  base base base tuned
  -------------------------------------------------------------------------------------
     scifact      0.693     0.6464     0.6945     0.6899     0.6913
    nfcorpus      0.344     0.3304     0.3949     0.3462     0.3497
        fiqa      0.356     0.3281     0.3283     0.3465     0.3421
     scidocs      0.154     0.1778     0.1875     0.1968     0.1986
     arguana      0.463     0.4188     0.4241     0.4220     0.4291

Key question: Does BGE-small tuned (33M) beat BGE-base untrained (110M)?
       scifact: sm_tuned=0.6945 vs base_base=0.6899 (+0.7%) -> sm tuned WINS
      nfcorpus: sm_tuned=0.3949 vs base_base=0.3462 (+14.1%) -> sm tuned WINS
          fiqa: sm_tuned=0.3283 vs base_base=0.3465 (-5.3%) -> base base wins
       scidocs: sm_tuned=0.1875 vs base_base=0.1968 (-4.7%) -> base base wins
       arguana: sm_tuned=0.4241 vs base_base=0.

In [ ]:
# Cell 5: Upload BGE-base tuned (only if results are good)
# !hf auth login
# !hf upload Stffens/bge-base-rrf-v1 experiments/models/bge-base-rrf --commit-message 'BGE-base fine-tuned with RRF disagreement'